# MinIO + MLflow: Machine Learning Pipeline

This notebook demonstrates:
1. Loading training data from MinIO
2. Training ML models with experiment tracking
3. Storing model artifacts in MinIO
4. Model versioning and deployment

## 1. Setup and Configuration

In [ ]:
import os
import mlflow
import mlflow.sklearn
from minio import Minio
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import json
from io import BytesIO

# MLflow configuration
mlflow.set_tracking_uri(os.environ.get("MLFLOW_TRACKING_URI", "http://mlflow:5000"))

# MinIO client
minio_client = Minio(
    "minio:9000",
    access_key=os.environ.get("AWS_ACCESS_KEY_ID", "myminio"),
    secret_key=os.environ.get("AWS_SECRET_ACCESS_KEY", "minio123"),
    secure=False
)

print(f"MLflow Tracking URI: {mlflow.get_tracking_uri()}")
print(f"MinIO connected: {minio_client.bucket_exists('ml-datasets')}")

## 2. Generate and Upload Sample Dataset to MinIO

In [ ]:
# Generate sample log classification dataset
np.random.seed(42)
n_samples = 1000

data = {
    'message_length': np.random.randint(10, 500, n_samples),
    'contains_error': np.random.randint(0, 2, n_samples),
    'contains_warning': np.random.randint(0, 2, n_samples),
    'hour_of_day': np.random.randint(0, 24, n_samples),
    'day_of_week': np.random.randint(0, 7, n_samples),
    'response_time_ms': np.random.exponential(100, n_samples),
    'request_count': np.random.poisson(10, n_samples),
    'severity': np.random.choice(['INFO', 'WARN', 'ERROR', 'DEBUG'], n_samples, p=[0.6, 0.2, 0.15, 0.05])
}

# Create target: predict if it's a critical log (needs attention)
data['is_critical'] = (
    (data['contains_error'] == 1) | 
    ((data['response_time_ms'] > 300) & (data['severity'] == 'ERROR'))
).astype(int)

df = pd.DataFrame(data)

# Upload to MinIO
csv_buffer = BytesIO(df.to_csv(index=False).encode())
minio_client.put_object(
    "ml-datasets",
    "log_classification/training_data.csv",
    csv_buffer,
    len(csv_buffer.getvalue()),
    content_type="text/csv"
)

print(f"Dataset uploaded to MinIO: ml-datasets/log_classification/training_data.csv")
print(f"Shape: {df.shape}")
print(f"Target distribution:\n{df['is_critical'].value_counts()}")

## 3. Load Data from MinIO for Training

In [ ]:
def load_dataset_from_minio(bucket: str, path: str) -> pd.DataFrame:
    """Load a CSV dataset from MinIO."""
    response = minio_client.get_object(bucket, path)
    return pd.read_csv(BytesIO(response.read()))

# Load training data
df = load_dataset_from_minio("ml-datasets", "log_classification/training_data.csv")

# Prepare features and target
# One-hot encode severity
df_encoded = pd.get_dummies(df, columns=['severity'], prefix='severity')

X = df_encoded.drop('is_critical', axis=1)
y = df_encoded['is_critical']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

## 4. Train Model with MLflow Tracking

In [ ]:
# Create or get experiment
experiment_name = "log-classification"
mlflow.set_experiment(experiment_name)

# Training function with MLflow tracking
def train_model(n_estimators: int, max_depth: int, min_samples_split: int):
    with mlflow.start_run() as run:
        # Log parameters
        mlflow.log_param("n_estimators", n_estimators)
        mlflow.log_param("max_depth", max_depth)
        mlflow.log_param("min_samples_split", min_samples_split)
        mlflow.log_param("data_source", "s3://ml-datasets/log_classification/training_data.csv")
        
        # Train model
        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            random_state=42
        )
        model.fit(X_train, y_train)
        
        # Predictions and metrics
        y_pred = model.predict(X_test)
        accuracy = accuracy_score(y_test, y_pred)
        
        # Log metrics
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("train_samples", len(X_train))
        mlflow.log_metric("test_samples", len(X_test))
        
        # Log model to MinIO
        mlflow.sklearn.log_model(
            model, 
            "model",
            registered_model_name="log-classifier"
        )
        
        # Log feature importance as artifact
        feature_importance = pd.DataFrame({
            'feature': X.columns,
            'importance': model.feature_importances_
        }).sort_values('importance', ascending=False)
        
        importance_path = "/tmp/feature_importance.csv"
        feature_importance.to_csv(importance_path, index=False)
        mlflow.log_artifact(importance_path)
        
        print(f"Run ID: {run.info.run_id}")
        print(f"Accuracy: {accuracy:.4f}")
        
        return run.info.run_id, accuracy

# Train with different hyperparameters
experiments = [
    {"n_estimators": 50, "max_depth": 5, "min_samples_split": 2},
    {"n_estimators": 100, "max_depth": 10, "min_samples_split": 2},
    {"n_estimators": 100, "max_depth": 10, "min_samples_split": 5},
    {"n_estimators": 200, "max_depth": 15, "min_samples_split": 2},
]

results = []
for params in experiments:
    run_id, accuracy = train_model(**params)
    results.append({"run_id": run_id, "accuracy": accuracy, **params})

# Show results
results_df = pd.DataFrame(results)
print("\n=== Experiment Results ===")
print(results_df.sort_values('accuracy', ascending=False))

## 5. Load Best Model and Make Predictions

In [ ]:
# Get the best run
best_run = results_df.loc[results_df['accuracy'].idxmax()]
print(f"Best model run ID: {best_run['run_id']}")
print(f"Best accuracy: {best_run['accuracy']:.4f}")

# Load model from MLflow (artifacts stored in MinIO)
model_uri = f"runs:/{best_run['run_id']}/model"
loaded_model = mlflow.sklearn.load_model(model_uri)

# Make predictions on new data
sample_data = pd.DataFrame({
    'message_length': [150, 400, 50],
    'contains_error': [1, 0, 0],
    'contains_warning': [0, 1, 0],
    'hour_of_day': [14, 3, 10],
    'day_of_week': [2, 5, 1],
    'response_time_ms': [450, 80, 120],
    'request_count': [15, 5, 8],
    'severity_DEBUG': [0, 0, 0],
    'severity_ERROR': [1, 0, 0],
    'severity_INFO': [0, 1, 1],
    'severity_WARN': [0, 0, 0]
})

predictions = loaded_model.predict(sample_data)
probabilities = loaded_model.predict_proba(sample_data)

print("\nPredictions for new logs:")
for i, (pred, prob) in enumerate(zip(predictions, probabilities)):
    status = "CRITICAL" if pred == 1 else "NORMAL"
    print(f"  Log {i+1}: {status} (confidence: {max(prob):.2%})")

## 6. Hyperparameter Tuning with Optuna

In [ ]:
import optuna
from sklearn.model_selection import cross_val_score

def objective(trial):
    # Hyperparameter search space
    n_estimators = trial.suggest_int("n_estimators", 50, 300)
    max_depth = trial.suggest_int("max_depth", 3, 20)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 10)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 5)
    
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        random_state=42
    )
    
    # Cross-validation
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
    return scores.mean()

# Run optimization
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20, show_progress_bar=True)

print(f"\nBest trial accuracy: {study.best_trial.value:.4f}")
print(f"Best parameters: {study.best_trial.params}")

# Train final model with best parameters and log to MLflow
with mlflow.start_run(run_name="optuna_best") as run:
    best_params = study.best_trial.params
    for key, value in best_params.items():
        mlflow.log_param(key, value)
    
    mlflow.log_param("optimization_method", "optuna")
    mlflow.log_param("n_trials", 20)
    
    final_model = RandomForestClassifier(**best_params, random_state=42)
    final_model.fit(X_train, y_train)
    
    accuracy = accuracy_score(y_test, final_model.predict(X_test))
    mlflow.log_metric("accuracy", accuracy)
    
    mlflow.sklearn.log_model(
        final_model,
        "model",
        registered_model_name="log-classifier-optimized"
    )
    
    print(f"\nFinal model accuracy: {accuracy:.4f}")
    print(f"Model logged to MLflow with run ID: {run.info.run_id}")

## 7. Feature Store Pattern with MinIO

In [ ]:
from datetime import datetime

class MinIOFeatureStore:
    """Simple feature store using MinIO."""
    
    def __init__(self, client: Minio, bucket: str = "feature-store"):
        self.client = client
        self.bucket = bucket
        
    def save_features(self, name: str, features: pd.DataFrame, metadata: dict = None):
        """Save features to MinIO with versioning."""
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        path = f"{name}/{timestamp}/features.parquet"
        
        # Save features
        buffer = BytesIO()
        features.to_parquet(buffer, index=False)
        buffer.seek(0)
        
        self.client.put_object(
            self.bucket,
            path,
            buffer,
            len(buffer.getvalue()),
            content_type="application/octet-stream"
        )
        
        # Save metadata
        if metadata:
            metadata["created_at"] = timestamp
            metadata["num_rows"] = len(features)
            metadata["columns"] = list(features.columns)
            
            meta_path = f"{name}/{timestamp}/metadata.json"
            meta_buffer = BytesIO(json.dumps(metadata).encode())
            
            self.client.put_object(
                self.bucket,
                meta_path,
                meta_buffer,
                len(meta_buffer.getvalue()),
                content_type="application/json"
            )
        
        print(f"Features saved to: {self.bucket}/{path}")
        return path
    
    def load_latest_features(self, name: str) -> pd.DataFrame:
        """Load the latest version of features."""
        objects = list(self.client.list_objects(self.bucket, prefix=f"{name}/", recursive=True))
        parquet_files = [o.object_name for o in objects if o.object_name.endswith(".parquet")]
        
        if not parquet_files:
            raise ValueError(f"No features found for: {name}")
        
        latest = sorted(parquet_files)[-1]
        response = self.client.get_object(self.bucket, latest)
        
        return pd.read_parquet(BytesIO(response.read()))

# Example usage
feature_store = MinIOFeatureStore(minio_client)

# Create engineered features
engineered_features = df_encoded.copy()
engineered_features['log_message_length'] = np.log1p(engineered_features['message_length'])
engineered_features['response_time_bucket'] = pd.cut(
    engineered_features['response_time_ms'],
    bins=[0, 100, 300, 1000, float('inf')],
    labels=['fast', 'normal', 'slow', 'critical']
)

# Save to feature store
feature_store.save_features(
    "log_features",
    engineered_features,
    metadata={
        "description": "Engineered features for log classification",
        "source": "log_classification/training_data.csv",
        "model": "log-classifier"
    }
)

# Load features
loaded_features = feature_store.load_latest_features("log_features")
print(f"\nLoaded features shape: {loaded_features.shape}")

## 8. Model Serving Preparation

In [ ]:
# Export model for serving
from mlflow.models import Model

# Get the latest registered model version
client = mlflow.tracking.MlflowClient()

try:
    latest_version = client.get_latest_versions("log-classifier-optimized", stages=["None"])[0]
    print(f"Latest model version: {latest_version.version}")
    print(f"Model source: {latest_version.source}")
    print(f"Run ID: {latest_version.run_id}")
    
    # Transition to Production (optional)
    # client.transition_model_version_stage(
    #     name="log-classifier-optimized",
    #     version=latest_version.version,
    #     stage="Production"
    # )
except Exception as e:
    print(f"Note: {e}")

print("\n=== Model Serving Options ===")
print("1. MLflow Model Serving:")
print("   mlflow models serve -m 'models:/log-classifier-optimized/latest' -p 5001")
print("\n2. Docker Container:")
print("   mlflow models build-docker -m 'models:/log-classifier-optimized/latest' -n log-classifier")
print("\n3. Kubernetes (KServe):")
print("   Model artifacts are stored in s3://mlflow-artifacts/ (MinIO)")